OANDA DOCUMENT: https://developer.oanda.com/rest-live-v20/instrument-df/

In [1]:
import requests
import defs
import pandas as pd

In [2]:
session = requests.Session()

we want to maintian "persistant connection" - using a session, to prevent creating a new request creating server 

In [3]:
instrument = "EUR_USD"
count = 10
granularity = "H1"

In [4]:
url = f"{defs.OANDA_URL}/instruments/{instrument}/candles"

In [5]:
url

'https://api-fxpractice.oanda.com/v3/instruments/EUR_USD/candles'

In [6]:
params = dict(
    count = count,
    granularity = granularity,
    price = "MBA"
)

In [7]:
params

{'count': 10, 'granularity': 'H1', 'price': 'MBA'}

In [8]:
response = session.get(url, params=params, headers=defs.SECURE_HEADER)

In [9]:
response.status_code

200

In [10]:
data = response.json()

In [11]:
data.keys()

dict_keys(['instrument', 'granularity', 'candles'])

In [12]:
len(data["candles"])

10

In [13]:
prices = ["mid", "bid", "ask"]
ohlc = ["o", "h", "l", "c"]

In [14]:
for price in prices:
    for oh in ohlc:
        print(f"{price}_{oh}")

mid_o
mid_h
mid_l
mid_c
bid_o
bid_h
bid_l
bid_c
ask_o
ask_h
ask_l
ask_c


In [15]:
data["candles"][0]["bid"]["o"] #This is how to access the data

'1.12297'

In [16]:
our_data = []
for candle in data["candles"]:
    if candle["complete"] == False:
        continue
    new_dict = {}
    new_dict["time"] = candle["time"]
    new_dict["volume"] = candle["volume"]

    for price in prices:
        for oh in ohlc:
            new_dict[f"{price}_{oh}"] = candle[price][oh]
    
    our_data.append(new_dict)
print(our_data)

[{'time': '2026-10-19T04:00:00.000000000Z', 'volume': 2841, 'mid_o': '1.12306', 'mid_h': '1.12348', 'mid_l': '1.12276', 'mid_c': '1.12324', 'bid_o': '1.12297', 'bid_h': '1.12341', 'bid_l': '1.12268', 'bid_c': '1.12317', 'ask_o': '1.12314', 'ask_h': '1.12356', 'ask_l': '1.12284', 'ask_c': '1.12332'}, {'time': '2026-10-19T05:00:00.000000000Z', 'volume': 3144, 'mid_o': '1.12324', 'mid_h': '1.12331', 'mid_l': '1.12252', 'mid_c': '1.12258', 'bid_o': '1.12316', 'bid_h': '1.12323', 'bid_l': '1.12243', 'bid_c': '1.12250', 'ask_o': '1.12332', 'ask_h': '1.12339', 'ask_l': '1.12260', 'ask_c': '1.12267'}, {'time': '2026-10-19T06:00:00.000000000Z', 'volume': 5159, 'mid_o': '1.12258', 'mid_h': '1.12278', 'mid_l': '1.12176', 'mid_c': '1.12277', 'bid_o': '1.12250', 'bid_h': '1.12269', 'bid_l': '1.12167', 'bid_c': '1.12269', 'ask_o': '1.12266', 'ask_h': '1.12286', 'ask_l': '1.12185', 'ask_c': '1.12285'}, {'time': '2026-10-19T07:00:00.000000000Z', 'volume': 9373, 'mid_o': '1.12278', 'mid_h': '1.12286', 

In [17]:
candle_df = pd.DataFrame(our_data)

In [18]:
candle_df

,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c
0,2026-10-19T04:00:00.000000000Z,2841,1.12306,1.12348,1.12276,1.12324,1.12297,1.12341,1.12268,1.12317,1.12314,1.12356,1.12284,1.12332
1,2026-10-19T05:00:00.000000000Z,3144,1.12324,1.12331,1.12252,1.12258,1.12316,1.12323,1.12243,1.12250,1.12332,1.12339,1.12260,1.12267
2,2026-10-19T06:00:00.000000000Z,5159,1.12258,1.12278,1.12176,1.12277,1.12250,1.12269,1.12167,1.12269,1.12266,1.12286,1.12185,1.12285
3,2026-10-19T07:00:00.000000000Z,9373,1.12278,1.12286,1.11866,1.11887,1.12270,1.12278,1.11858,1.11879,1.12285,1.12294,1.11873,1.11895
4,2026-10-19T08:00:00.000000000Z,7962,1.11889,1.12172,1.11889,1.12128,1.11881,1.12164,1.11881,1.12119,1.11897,1.12179,1.11897,1.12136
5,2026-10-19T09:00:00.000000000Z,6222,1.12128,1.12178,1.12066,1.12068,1.12120,1.12170,1.12058,1.12060,1.12136,1.12186,1.12075,1.12076
6,2026-10-19T10:00:00.000000000Z,7635,1.12068,1.12204,1.12051,1.12174,1.12059,1.12197,1.12044,1.12167,1.12076,1.12212,1.12058,1.12182
7,2026-10-19T11:00:00.000000000Z,6869,1.12175,1.12312,1.12174,1.12288,1.12167,1.12304,1.12166,1.12280,1.12183,1.12321,1.12182,1.12295
8,2026-10-19T12:00:00.000000000Z,7806,1.12288,1.12290,1.12116,1.12236,1.12281,1.12283,1.12108,1.12228,1.12295,1.12298,1.12123,1.12245


In [19]:
candle_df.to_pickle("candle_df.pkl")

In [20]:
test_df = pd.read_pickle("candle_df.pkl")

In [21]:
test_df

,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c
0,2026-10-19T04:00:00.000000000Z,2841,1.12306,1.12348,1.12276,1.12324,1.12297,1.12341,1.12268,1.12317,1.12314,1.12356,1.12284,1.12332
1,2026-10-19T05:00:00.000000000Z,3144,1.12324,1.12331,1.12252,1.12258,1.12316,1.12323,1.12243,1.12250,1.12332,1.12339,1.12260,1.12267
2,2026-10-19T06:00:00.000000000Z,5159,1.12258,1.12278,1.12176,1.12277,1.12250,1.12269,1.12167,1.12269,1.12266,1.12286,1.12185,1.12285
3,2026-10-19T07:00:00.000000000Z,9373,1.12278,1.12286,1.11866,1.11887,1.12270,1.12278,1.11858,1.11879,1.12285,1.12294,1.11873,1.11895
4,2026-10-19T08:00:00.000000000Z,7962,1.11889,1.12172,1.11889,1.12128,1.11881,1.12164,1.11881,1.12119,1.11897,1.12179,1.11897,1.12136
5,2026-10-19T09:00:00.000000000Z,6222,1.12128,1.12178,1.12066,1.12068,1.12120,1.12170,1.12058,1.12060,1.12136,1.12186,1.12075,1.12076
6,2026-10-19T10:00:00.000000000Z,7635,1.12068,1.12204,1.12051,1.12174,1.12059,1.12197,1.12044,1.12167,1.12076,1.12212,1.12058,1.12182
7,2026-10-19T11:00:00.000000000Z,6869,1.12175,1.12312,1.12174,1.12288,1.12167,1.12304,1.12166,1.12280,1.12183,1.12321,1.12182,1.12295
8,2026-10-19T12:00:00.000000000Z,7806,1.12288,1.12290,1.12116,1.12236,1.12281,1.12283,1.12108,1.12228,1.12295,1.12298,1.12123,1.12245
